In [1]:
import pandas as pd
import numpy as np

print("Notebook environment working!")

Notebook environment working!


# 1. Data Sources and Unit of Analysis
## Unit of Analysis

One row represents one debt collection matter/account at a defined point in time.

## Modelling Objective

The objective is to predict whether a debt collection matter will result in a payment (penetration), using information available before the payment outcome occurs.

The model should identify matters with a higher likelihood of payment and provide insights that can support more effective collection strategies.

## Data Sources

The project uses multiple data sources related to debt collection activity, including:

- Matters
- CallHistory
- PTP
- Payments
- Enriched datasets

These sources contain information about collection matters, contact activity, promises to pay, and payment outcomes.

## Inspecting the Raw Data

In [2]:
from pathlib import Path

raw_data_path = Path("../data/raw")

files = list(raw_data_path.iterdir())

for file in files:
    print(file.name)

PTP.xlsx
Matters.xlsx
Payments.xlsx
UKZN_EnrichedData2.xlsx
UKZN_EnrichedData1.xlsx
CallHistory.xlsx


In [3]:
for file in files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:.2f} MB")

PTP.xlsx: 0.72 MB
Matters.xlsx: 3.23 MB
Payments.xlsx: 0.24 MB
UKZN_EnrichedData2.xlsx: 51.00 MB
UKZN_EnrichedData1.xlsx: 48.33 MB
CallHistory.xlsx: 27.01 MB


In [5]:
import pandas as pd

for file in files:
    excel_file = pd.ExcelFile(file)
    print(f"\n{file.name}")
    print(excel_file.sheet_names)


PTP.xlsx
['Data Description', 'Data']

Matters.xlsx
['Data description', 'Data']

Payments.xlsx
['Data description', 'Data']

UKZN_EnrichedData2.xlsx
['UKZN_EnrichedData2']

UKZN_EnrichedData1.xlsx
['UKZN_EnrichedData']

CallHistory.xlsx
['Data Description', 'Data']


In [6]:
matters_description = pd.read_excel(
    raw_data_path / "Matters.xlsx",
    sheet_name="Data description"
)

matters_description.head()

,Field,Description
0,M_IDX,Unique Identifier
1,HandoverDate,The date the client sent us the outstanding ac...
2,CurrentStatusID,We use IDs of various numbers that relate to a...
3,CurrentStatus,"Status of the debtor account e.g. on hold, clo..."
4,PreviousStatusID,ID relating to the PreviousStatus description ...


In [7]:
matters_description

,Field,Description
0,M_IDX,Unique Identifier
1,HandoverDate,The date the client sent us the outstanding ac...
2,CurrentStatusID,We use IDs of various numbers that relate to a...
3,CurrentStatus,"Status of the debtor account e.g. on hold, clo..."
4,PreviousStatusID,ID relating to the PreviousStatus description ...
5,PreviousStatus,Used to see how an account potentially changed...
6,FirstPaymentDate,Date first payment was made on the outstanding...
7,ActivationPeriod,Number of months between a matter being handed...
8,OpeningBalance,The amount the account owed when we received i...
9,CurrentBalance,The value of the matters account as of today i...


In [ ]:
matters = pd.read_excel(
    raw_data_path / "Matters.xlsx",
    sheet_name="Data"
)

print(f"Rows: {matters.shape[0]:,}")
print(f"Columns: {matters.shape[1]:,}")

matters.head()

Rows: 56,179
Columns: 12


,M_IDX,HandoverDate,CurrentStatusID,CurrentStatus,PreviousStatusID,PreviousStatus,FirstPaymentDate,ActivationPeriod,OpeningBalance,CurrentBalance,Industry,DateCreated
0,862557,2025-02-11,5,Closed,NaN,NaN,2025-02-27,0.0,202.93,0.00,Medical,2025-05-23
1,862558,2025-02-11,5,Closed,NaN,NaN,2025-02-19,0.0,771.43,0.00,Medical,2025-05-23
2,862559,2025-02-11,5,Closed,NaN,NaN,2025-02-20,0.0,221.54,0.00,Medical,2025-05-23
3,865730,2025-02-12,5,Closed,NaN,NaN,2025-03-10,1.0,1539.77,-300.00,Medical,2025-05-23
4,865733,2025-02-12,5,Closed,NaN,NaN,2025-02-28,0.0,603.58,0.42,Medical,2025-05-23


In [9]:
matters.dtypes

M_IDX                        int64
HandoverDate        datetime64[us]
CurrentStatusID              int64
CurrentStatus                  str
PreviousStatusID           float64
PreviousStatus                 str
FirstPaymentDate    datetime64[us]
ActivationPeriod           float64
OpeningBalance             float64
CurrentBalance             float64
Industry                       str
DateCreated         datetime64[us]
dtype: object

In [10]:
missing = matters.isna().sum()

missing_percentage = (missing / len(matters) * 100).round(2)

pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage
}).sort_values("Missing_Percentage", ascending=False)

,Missing_Count,Missing_Percentage
PreviousStatusID,55027,97.95
PreviousStatus,55027,97.95
FirstPaymentDate,52408,93.29
ActivationPeriod,52408,93.29
CurrentBalance,939,1.67
M_IDX,0,0.00
HandoverDate,0,0.00
CurrentStatusID,0,0.00
CurrentStatus,0,0.00
OpeningBalance,0,0.00


In [11]:
print("Missing FirstPaymentDate:")
print(matters["FirstPaymentDate"].isna().value_counts())

print("\nCurrent Status:")
print(matters["CurrentStatus"].value_counts())

Missing FirstPaymentDate:
FirstPaymentDate
True     52408
False     3771
Name: count, dtype: int64

Current Status:
CurrentStatus
Attempting PTP         26474
Closed                 15882
New Instruction         6341
Payment Arrangement     3633
Broken PTP              2817
Re-opened                292
Authentication           271
In Progress              232
On Hold                  160
Follow-up PTP             70
Request for Closure        6
Dispute                    1
Name: count, dtype: int64


## Inspecting Payment Records

The Payments table contains transaction-level information about payments made against collection matters.

We will inspect its structure and identify the fields that can be used to construct the payment outcome.

In [15]:
payments_description = pd.read_excel(
    raw_data_path / "Payments.xlsx",
    sheet_name="Data description"
)

payments_description.head()

,Field,Description
0,M_IDX,Unique Identifier
1,AmountPaid,"Positive for received payments, negative for r..."
2,IsPayedAtClient,Indicator if the matter paid at the client dir...
3,IsDebitOrderPayment,Indicator if the matter paid by debit order (i...
4,PaymentMethodID,ID related to the payment method column


In [17]:
payments_description

,Field,Description
0,M_IDX,Unique Identifier
1,AmountPaid,"Positive for received payments, negative for r..."
2,IsPayedAtClient,Indicator if the matter paid at the client dir...
3,IsDebitOrderPayment,Indicator if the matter paid by debit order (i...
4,PaymentMethodID,ID related to the payment method column
5,PaymentDescription,Type of payment method used and indicated by t...
6,Agent,This links the payment to an agent who is link...


In [21]:
payments = pd.read_excel(
    raw_data_path / "Payments.xlsx",
    sheet_name="Data"
)

print(f"Rows: {payments.shape[0]:,}")
print(f"Columns: {payments.shape[1]:,}")

payments.head()

Rows: 6,016
Columns: 8


,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
0,817511,2025-04-11,2215.00,0,0,1,Direct Deposit,73867
1,817515,2025-04-23,1164.84,1,0,0,Unspecified,73868
2,817516,2025-02-06,479.00,1,0,0,Unspecified,-1
3,817517,2025-02-12,1227.88,1,0,0,Unspecified,-1
4,817519,2025-04-08,847.44,1,0,0,Unspecified,73868


In [19]:
payments.dtypes

M_IDX                           int64
PaymentDate            datetime64[us]
AmountPaid                    float64
IsPayedAtClient                 int64
IsDebitOrderPayment             int64
PaymentMethodID                 int64
PaymentDescription                str
AgentID                         int64
dtype: object

In [22]:
missing = payments.isna().sum()

missing_percentage = (missing / len(payments) * 100).round(2)

pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage
}).sort_values("Missing_Percentage", ascending=False)

,Missing_Count,Missing_Percentage
M_IDX,0,0.0
PaymentDate,0,0.0
AmountPaid,0,0.0
IsPayedAtClient,0,0.0
IsDebitOrderPayment,0,0.0
PaymentMethodID,0,0.0
PaymentDescription,0,0.0
AgentID,0,0.0


## Inspecting Call History Records

The CallHistory table contains call-level information associated with collection matters.

We will inspect its structure and identify the fields that describe contact attempts and collection activity.

In [25]:
callHistory_description = pd.read_excel(
    raw_data_path / "CallHistory.xlsx",
    sheet_name="Data Description"
)

callHistory_description.head()

,Field,Description
0,M_IDX,Unique Identifier
1,AgentID,This links the call to an agent who is linked ...
2,HistoryID,ID that links call history data to the PTP th...
3,CallDate,The date that the call was made to the matter
4,MinutesOfCall,The number of minutes a call was connected


In [26]:
callHistory_description

,Field,Description
0,M_IDX,Unique Identifier
1,AgentID,This links the call to an agent who is linked ...
2,HistoryID,ID that links call history data to the PTP th...
3,CallDate,The date that the call was made to the matter
4,MinutesOfCall,The number of minutes a call was connected
5,HasRPC,RPC- Right Party Contact; This refers if the c...
6,CallTypeID,ID linked to the type of call that was made
7,CallType,The description for the call type linked to Ca...
8,PTPCreateIndicator,Indicator whether a PTP was created from the c...
9,TimeCallStart,Exact time the call was started.


In [27]:
callHistory = pd.read_excel(
    raw_data_path / "CallHistory.xlsx",
    sheet_name="Data"
)

print(f"Rows: {callHistory.shape[0]:,}")
print(f"Columns: {callHistory.shape[1]:,}")

callHistory.head()

Rows: 430,197
Columns: 12


,M_IDX,AgentID,HistoryID,CallDate,MinutesOfCall,HasRPC,CallTypeID,CallType,PTPCreateIndicator,TimeCallStart,TimeCallConfirmRPC,TimeCallEnded
0,820331,73868,26809600,2025-02-06,1.0,NaN,22.0,Outbound call,NaN,2025-02-06 08:27:30.053,NaN,2025-02-06 08:27:42.980
1,820331,73868,26809685,2025-02-06,1.0,NaN,22.0,Outbound call,NaN,2025-02-06 08:28:11.107,NaN,2025-02-06 08:28:36.377
2,820331,73868,26809786,2025-02-06,1.0,NaN,22.0,Outbound call,NaN,2025-02-06 08:29:13.637,NaN,2025-02-06 08:29:17.713
3,820331,73868,26809842,2025-02-06,1.0,NaN,22.0,Outbound call,NaN,2025-02-06 08:29:40.383,NaN,2025-02-06 08:30:03.303
4,820331,73868,26863284,2025-02-06,2.0,1.0,22.0,Outbound call,1.0,2025-02-06 11:55:22.893,2025-02-06 11:57:00.150,2025-02-06 11:57:18.570


In [28]:
callHistory.dtypes

M_IDX                          int64
AgentID                        int64
HistoryID                      int64
CallDate              datetime64[us]
MinutesOfCall                float64
HasRPC                       float64
CallTypeID                   float64
CallType                         str
PTPCreateIndicator           float64
TimeCallStart                    str
TimeCallConfirmRPC               str
TimeCallEnded                    str
dtype: object

In [29]:
missing = callHistory.isna().sum()

missing_percentage = (missing / len(callHistory) * 100).round(2)

pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage
}).sort_values("Missing_Percentage", ascending=False)

,Missing_Count,Missing_Percentage
PTPCreateIndicator,421988,98.09
TimeCallConfirmRPC,412781,95.95
HasRPC,347689,80.82
MinutesOfCall,226131,52.56
TimeCallEnded,226131,52.56
TimeCallStart,221181,51.41
CallTypeID,80073,18.61
M_IDX,0,0.00
AgentID,0,0.00
HistoryID,0,0.00


## Inspecting PTP Records

The PTP table contains information about promise-to-pay (PTP) arrangements associated with collection matters.

We will inspect its structure and identify the fields that describe payment promises and PTP activity.

In [31]:
ptp_description = pd.read_excel(
    raw_data_path / "PTP.xlsx",
    sheet_name="Data Description"
)

ptp_description.head()

,Field,Description
0,M_IDX,Unique Identifier
1,PTPCreateDate,PTP - Promise To Pay; This is the day an agent...
2,PaymentMethodID,ID related to the payment method column
3,PaymentDescription,Type of payment method used and indicated by t...
4,PaymentFrequencyID,ID related to the payment frequency column


In [32]:
ptp_description

,Field,Description
0,M_IDX,Unique Identifier
1,PTPCreateDate,PTP - Promise To Pay; This is the day an agent...
2,PaymentMethodID,ID related to the payment method column
3,PaymentDescription,Type of payment method used and indicated by t...
4,PaymentFrequencyID,ID related to the payment frequency column
5,PaymentFrequency,Type of payment frequency indicated in the PTP...
6,FirstPaymentAmount,This is the amount the matter paid for their f...
7,FirstPaymentDate,This is the date the first payment was made an...
8,MonthlyPaymentAmount,This is the amount the matter pays according t...
9,MonthlyPaymentStartDate,This is the first date the recurring payment p...


In [34]:
ptp = pd.read_excel(
    raw_data_path / "PTP.xlsx",
    sheet_name = "Data"
)

print(f"Rows: {ptp.shape[0]:,}")
print(f"Columns: {ptp.shape[1]:,}")

ptp.head()

Rows: 9,402
Columns: 16


,M_IDX,PTPCreateDate,PaymentMethodID,PaymentDescription,PaymentFrequencyID,PaymentFrequency,FirstPaymentAmount,FirstPaymentDate,MonthlyPaymentAmount,MonthlyPaymentStartDate,DebitAmount,CreditAmount,ProjectionPaymentDate,ProjectionDescriptionID,ProjectionDescription,HistoryID
0,893941,2025-02-24,1,Direct Deposit,5,NaN,403.60,2025-02-25,0.0,NaT,NaN,403.60,2025-02-25,7,Payment Settlement,32165892
1,868821,2025-02-25,1,Direct Deposit,5,NaN,484.75,2025-03-20,0.0,NaT,NaN,484.75,2025-03-20,7,Payment Settlement,32265243
2,868806,2025-02-25,1,Direct Deposit,5,NaN,1101.26,2025-02-28,0.0,NaT,NaN,1101.26,2025-02-28,7,Payment Settlement,32265505
3,868742,2025-02-25,1,Direct Deposit,5,NaN,100.00,2025-02-27,0.0,NaT,NaN,100.00,2025-02-27,3,Payment Initial,32266606
4,896115,2025-02-28,1,Direct Deposit,3,Monthly,100.00,2025-03-05,100.0,2025-04-04,NaN,100.00,2025-03-05,3,Payment Initial,33263458


In [35]:
ptp.dtypes

M_IDX                               int64
PTPCreateDate              datetime64[us]
PaymentMethodID                     int64
PaymentDescription                    str
PaymentFrequencyID                  int64
PaymentFrequency                      str
FirstPaymentAmount                float64
FirstPaymentDate           datetime64[us]
MonthlyPaymentAmount              float64
MonthlyPaymentStartDate    datetime64[us]
DebitAmount                       float64
CreditAmount                      float64
ProjectionPaymentDate      datetime64[us]
ProjectionDescriptionID             int64
ProjectionDescription                 str
HistoryID                           int64
dtype: object

In [36]:
missing = ptp.isna().sum()

missing_percentage = (missing / len(ptp) * 100).round(2)

pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage
}).sort_values("Missing_Percentage", ascending=False)

,Missing_Count,Missing_Percentage
DebitAmount,9391,99.88
FirstPaymentDate,6174,65.67
PaymentFrequency,1835,19.52
MonthlyPaymentStartDate,1834,19.51
FirstPaymentAmount,161,1.71
M_IDX,0,0.00
PTPCreateDate,0,0.00
PaymentMethodID,0,0.00
PaymentDescription,0,0.00
PaymentFrequencyID,0,0.00


**## Inspecting Enriched Data**

The Enriched dataset contains additional information about collection matters across a large number of variables.

We will inspect its structure and identify the fields that provide additional information about the matters.
